Report

In this experiment, we used the Exorde social media one-month 2024 dataset and accessed it via HuggingFace streaming mode to avoid downloading the full large-scale corpus. Using streaming with shuffling, we subsampled 300 social media posts for lightweight exploratory analysis. This approach enables fast experimentation while preserving the heterogeneity of the original data.
Automated annotations.

Several automated annotations were applied, following the lecture’s emphasis on weak supervision and surface-level text analytics. First, we extracted surface features from each post, including character length, word count, and the presence and frequency of URLs, mentions, hashtags, and emojis. The sampled posts are moderately long on average (≈41 words), while explicit social-media markers such as URLs (0.3%), mentions (2.3%), hashtags (6.7%), and emojis (very rare) appear infrequently in this subset.

Second, we discretized sentiment scores into three weak labels (negative, neutral, positive). The sentiment distribution is slightly skewed toward polarity: 149 posts are labeled positive, 116 negative, and only 35 neutral. This suggests that neutral sentiment is underrepresented, either due to the nature of online discourse or due to the conservative thresholding of the sentiment model.

Third, we derived source-domain labels by extracting URL domains. Nearly all posts (299/300) contain no external links, indicating that the subsample is dominated by conversational or opinion-style content rather than link-sharing behavior.

Fourth, we analyzed pre-computed metadata provided by the dataset. Language distribution shows strong dominance of English (192 posts), followed by Spanish and Portuguese, reflecting multilingual but English-centered discourse. The primary theme annotation highlights “People,” “Entertainment,” “Politics,” and “Sports” as the most frequent categories, demonstrating topical diversity even in a small sample.

Finally, we implemented a rule-based topic annotation using keyword dictionaries (e.g., politics, finance, sports, technology). Most posts (217) fall into an “other” category, while technology-related content appears most frequently among matched topics. This confirms that simple keyword rules provide coarse but interpretable thematic signals, albeit with limited coverage.

In [1]:
from datasets import load_dataset
import random

ds = load_dataset("Exorde/exorde-social-media-one-month-2024", split="train", streaming=True)


samples = list(ds.take(300))


ds_shuf = ds.shuffle(buffer_size=50_000, seed=42)
samples = list(ds_shuf.take(300))

print(samples[0].keys())
print(samples[0]["language"], samples[0]["original_text"][:120])


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/543 [00:00<?, ?it/s]

dict_keys(['date', 'original_text', 'url', 'author_hash', 'language', 'primary_theme', 'english_keywords', 'sentiment', 'main_emotion', 'secondary_themes'])
es 'Euskararen Eguna' pone cada 3 de diciembre la reivindicación de la lengua en la calle. Por la mañana (12.00) el alumn


In [2]:
import pandas as pd


In [3]:
df = pd.DataFrame(samples)
print(df.columns)
df.head(3)

Index(['date', 'original_text', 'url', 'author_hash', 'language',
       'primary_theme', 'english_keywords', 'sentiment', 'main_emotion',
       'secondary_themes'],
      dtype='object')


,date,original_text,url,author_hash,language,primary_theme,english_keywords,sentiment,main_emotion,secondary_themes
0,2024-12-02T18:29:10.000Z,'Euskararen Eguna' pone cada 3 de diciembre la...,https://www.diariovasco.com/alto-deba/onati/ch...,53733b4699d9672365e02ebb68e4423902d4a23f,es,Entertainment,"work, theatre work, centers, eguna, company, e...",0.46,joy,"[3, 14, 7]"
1,2024-12-02T19:35:44.000Z,The most corrupt regime ever to hold power in ...,https://x.com/Phuck3D/status/1863668389828276627,None,en,Politics,"corrupt, hold power, power, hold, USA, the, re...",-0.77,annoyance,"[3, 1, 11]"
2,2024-12-02T19:33:35.000Z,I knew it couldn't of just been a few doctors ...,https://x.com/penguinmelonskz/status/186366785...,None,en,Health,"interesting, pharmaceutical range, bacterial, ...",0.15,neutral,"[9, 4, 7]"


In [5]:
import re

URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#\w+")
EMOJI_RE = re.compile(
    "["                     # very rough emoji range
    "\U0001F300-\U0001F5FF"
    "\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF"
    "\U0001F700-\U0001F77F"
    "\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF"
    "\U0001F900-\U0001F9FF"
    "\U0001FA00-\U0001FA6F"
    "\U0001FA70-\U0001FAFF"
    "]+", flags=re.UNICODE
)

def get_text(row):
    for c in ["original_text", "text", "content"]:
        if c in row and isinstance(row[c], str) and row[c].strip():
            return row[c]
    return ""

df["text"] = df.apply(get_text, axis=1)

df["char_len"] = df["text"].str.len()
df["word_len"] = df["text"].str.split().str.len()

df["n_urls"] = df["text"].apply(lambda s: len(URL_RE.findall(s)))
df["n_mentions"] = df["text"].apply(lambda s: len(MENTION_RE.findall(s)))
df["n_hashtags"] = df["text"].apply(lambda s: len(HASHTAG_RE.findall(s)))
df["n_emojis"] = df["text"].apply(lambda s: len(EMOJI_RE.findall(s)))

df["has_url"] = df["n_urls"] > 0
df["has_mention"] = df["n_mentions"] > 0
df["has_hashtag"] = df["n_hashtags"] > 0
df["has_emoji"] = df["n_emojis"] > 0

df[["text","char_len","word_len","n_urls","n_mentions","n_hashtags","n_emojis"]].head(5)


,text,char_len,word_len,n_urls,n_mentions,n_hashtags,n_emojis
0,'Euskararen Eguna' pone cada 3 de diciembre la...,275,43,0,0,0,0
1,The most corrupt regime ever to hold power in ...,59,12,0,0,0,1
2,I knew it couldn't of just been a few doctors ...,244,45,0,0,0,0
3,God doesn't just want us to have faith in Jesu...,160,32,0,0,0,0
4,my baby jumped in her sleep Aisha said that’s ...,131,19,0,0,0,1


In [7]:
if "sentiment" in df.columns:
    print(df["sentiment"].describe())

    def sentiment_bucket(x):
        if pd.isna(x):
            return "unknown"
        
        if x <= -0.05:
            return "negative"
        elif x >= 0.05:
            return "positive"
        else:
            return "neutral"

    df["sentiment_label"] = df["sentiment"].apply(sentiment_bucket)
    print(df["sentiment_label"].value_counts())
else:
    print("No 'sentiment' column found.")


count    300.000000
mean       0.037883
std        0.401962
min       -0.940000
25%       -0.250000
50%        0.040000
75%        0.313750
max        0.930000
Name: sentiment, dtype: float64
sentiment_label
positive    149
negative    116
neutral      35
Name: count, dtype: int64


In [ ]:
from urllib.parse import urlparse
# source label
def extract_domains(text):
    urls = URL_RE.findall(text or "")
    domains = []
    for u in urls:
        try:
            d = urlparse(u).netloc.lower()
            d = d.replace("www.", "")
            if d:
                domains.append(d)
        except Exception:
            pass
    return domains

df["domains"] = df["text"].apply(extract_domains)
df["main_domain"] = df["domains"].apply(lambda ds: ds[0] if len(ds) else "none")

df["main_domain"].value_counts().head(10)


main_domain
none                      299
meticulousresearch.com      1
Name: count, dtype: int64

In [9]:
for col in ["language", "primary_theme", "main_emotion"]:
    if col in df.columns:
        print("\n", col)
        print(df[col].value_counts().head(10))



 language
language
en    192
es     43
pt     22
de      7
tr      6
fr      5
fa      4
it      3
ru      2
ja      2
Name: count, dtype: int64

 primary_theme
primary_theme
People           57
Entertainment    51
Politics         40
Sports           36
Technology       26
Science          16
Health           14
Business         11
Environment      10
Social           10
Name: count, dtype: int64

 main_emotion
main_emotion
neutral        191
approval        21
admiration      11
curiosity       11
annoyance        9
gratitude        7
disapproval      6
joy              5
desire           5
excitement       5
Name: count, dtype: int64


In [10]:
TOPIC_RULES = {
    "politics": ["election", "vote", "government", "president", "parliament", "policy"],
    "finance": ["stock", "market", "inflation", "bank", "crypto", "bitcoin", "rates"],
    "sports": ["game", "match", "goal", "team", "league", "nba", "fifa"],
    "tech": ["ai", "model", "openai", "chip", "gpu", "software", "python"],
}

def rule_topic(text):
    t = (text or "").lower()
    hits = []
    for topic, kws in TOPIC_RULES.items():
        if any(kw in t for kw in kws):
            hits.append(topic)
    if len(hits) == 0:
        return "other"
    if len(hits) == 1:
        return hits[0]
    return "multi:" + ",".join(hits)

df["rule_topic"] = df["text"].apply(rule_topic)
df["rule_topic"].value_counts().head(10)


rule_topic
other                                 217
tech                                   49
multi:sports,tech                      10
sports                                  7
politics                                4
multi:politics,tech                     4
multi:finance,tech                      4
finance                                 2
multi:finance,sports,tech               1
multi:politics,finance,sports,tech      1
Name: count, dtype: int64

In [11]:
summary = {
    "N": len(df),
    "pct_has_url": df["has_url"].mean(),
    "pct_has_hashtag": df["has_hashtag"].mean(),
    "pct_has_mention": df["has_mention"].mean(),
    "avg_word_len": df["word_len"].mean(),
    "top_domains": df["main_domain"].value_counts().head(5).to_dict(),
}
summary


{'N': 300,
 'pct_has_url': np.float64(0.0033333333333333335),
 'pct_has_hashtag': np.float64(0.06666666666666667),
 'pct_has_mention': np.float64(0.023333333333333334),
 'avg_word_len': np.float64(41.193333333333335),
 'top_domains': {'none': 299, 'meticulousresearch.com': 1}}